# 6. Transformer models for text processing

---

In [12]:
import numpy as np
import tensorflow as tf
import keras
from keras import layers, ops
import random

--- 

## Load and Preprocess the Data
We load the dataset from a text file that contains English–Finnish sentence pairs.

Each line in the file has sentences separated by a tab (`\t`).
We read the file, split it into lines, and process them one by one.


In [2]:
text_file = "fin-eng/fin.txt"

with open(text_file, encoding='utf-8') as f:
    lines = f.read().split("\n")[:-1]

text_pairs = []
for line in lines:
    parts = line.split("\t") # splitting into 3 parts
    if len(parts) >= 2:
        english, finnish = parts[0], parts[1] # assign values into english and finnish
        finnish = "[start] " + finnish + " [end]"
        text_pairs.append((english, finnish))

random.shuffle(text_pairs) # randomly shuffle the dataset to avoid any ordering bias
num_val_samples = int(0.15 * len(text_pairs)) # calculate the numbers of validation samples
num_train_samples = len(text_pairs) - 2 * num_val_samples

# splitting datasets
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples : num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples :]

---

## Vectorization

In this step, text is converted into numbers so the model can process it.

We use TextVectorization for both English and Finnish, limiting the vocabulary to 15,000 words and ensuring all sequences have the same length.

The vectorizers are fitted only on the training data.

The dataset is then formatted for the model. English is used as the encoder input, while Finnish is split into a decoder input without the last token and a target without the first token.

Finally, the data is **batched**, **shuffled**, and **optimized** using prefetch and cache.

In [3]:
vocab_size = 15000
sequence_length = 20
batch_size = 64

eng_vectorization = layers.TextVectorization(
    max_tokens=vocab_size, output_mode="int", output_sequence_length=sequence_length,
)

fin_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
)

train_eng_texts = [pair[0] for pair in train_pairs]
eng_vectorization.adapt(train_eng_texts)

train_fin_texts = [pair[1] for pair in train_pairs]
fin_vectorization.adapt(train_fin_texts)

# format dataset into encoder and decoder inputs
def format_dataset(eng, fin):
    eng = eng_vectorization(eng)
    fin = fin_vectorization(fin)
    return ({
        "encoder_inputs": eng,
        "decoder_inputs": fin[:, :-1],
    }, fin[:, 1:])

# create a tensorflow dataset from text pairs
def make_dataset(pairs):
    # unzip list of tuples into two separate lists
    eng_texts, fin_texts = zip(*pairs)

    eng_texts = list(eng_texts)
    fin_texts = list(fin_texts)

    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, fin_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset)

# shuffle dataset for randomness values, preload batches for speed, and cache results
    return dataset.shuffle(2048).prefetch(16).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

---

## Transformer Components
We implement PositionalEmbedding, TransformerEncoder, and TransformerDecoder. For translation, the decoder uses two attention layers: causal self-attention and cross-attention.

### Positional Embedding 
We create a layer that combines word embeddings and position information. Token embeddings convert words into vectors, and position embeddings provide information about the order of words in the sentence. In the call function, these two embeddings are added together to form the final representation. Padding tokens (value 0) are masked out, and get_config is used to make sure the layer can be saved and loaded correctly.

In [4]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)

        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim, mask_zero=True
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        length = ops.shape(inputs)[-1]
        positions = ops.arange(0, length, 1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config

### Transformer Encoder

The encoder processes input sequences using self-attention. It allows each word to look at other words in the sentence to understand context. After attention, the result goes through a small feed-forward network. Residual connections and layer normalization are used to keep training stable. Padding tokens are ignored using a mask, and `get_config` is used so the layer can be saved and loaded properly.

In [5]:
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"), 
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        if mask is not None:
            padding_mask = ops.cast(mask[:, None, :], dtype="int32")
        else:
            padding_mask = None

        attention_output = self.attention(
            query=inputs, value=inputs, key=inputs, attention_mask=padding_mask
        )
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

### Transformer Decoder
The decoder generates the output sequence step by step. It first uses self-attention with a causal mask so it can only look at previous tokens. Then it uses cross-attention to look at the encoder output and get information from the input sentence. After that, the result goes through a feed-forward network. Residual connections and layer normalization are used after each step to keep training stable. Masking is used to prevent looking at future tokens and to handle padding, and `get_config` allows the layer to be saved and loaded properly.

In [6]:
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.latent_dim = latent_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(latent_dim, activation="relu"), 
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, encoder_outputs, mask=None):
        causal_mask = self.get_causal_attention_mask(inputs)

        # Combine the padding mask with the causal mask
        if mask is not None:
            padding_mask = ops.cast(mask[:, None, :], dtype="int32")
            padding_mask = ops.minimum(padding_mask, causal_mask)
        else:
            padding_mask = causal_mask

        # first attention (self-attention on decoder inputs)
        attention_output_1 = self.attention_1(
            query=inputs, value=inputs, key=inputs, 
            attention_mask=padding_mask
        )
        out_1 = self.layernorm_1(inputs + attention_output_1)

        # second attention (cross-attention with encoder outputs)
        attention_output_2 = self.attention_2(
            query=out_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=None,
        )
        out_2 = self.layernorm_2(out_1 + attention_output_2)

        proj_output = self.dense_proj(out_2)
        return self.layernorm_3(out_2 + proj_output)

    def get_causal_attention_mask(self, inputs):
        input_shape = ops.shape(inputs)
        sequence_length = input_shape[1]
        i = ops.arange(sequence_length)[:, None]
        j = ops.arange(sequence_length)
        mask = ops.cast(i >= j, dtype="int32")
        return ops.reshape(mask, (1, sequence_length, sequence_length))

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "latent_dim": self.latent_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

---

## Build the Model
A Transformer model is built using an encoder and a decoder. The encoder processes the input sentence and produces a representation of it, while the decoder generates the output sentence using both previous tokens and the encoder output. A dropout layer is applied to reduce overfitting, and the final softmax layer is used to predict the next word in the vocabulary. The model is compiled using the RMSprop optimizer and trained with sparse categorical cross-entropy loss, with accuracy used as a metric.

In [7]:
embed_dim = 64
latent_dim = 256
num_heads = 4

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="encoder_inputs")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, latent_dim, num_heads)(x)
encoder = keras.Model(encoder_inputs, encoder_outputs)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="decoder_inputs")
encoded_seq_inputs = keras.Input(shape=(None, embed_dim), name="decoder_state_inputs")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, latent_dim, num_heads)(x, encoded_seq_inputs)
x = layers.Dropout(0.5)(x)
decoder_outputs = layers.Dense(vocab_size, activation="softmax")(x)
decoder = keras.Model([decoder_inputs, encoded_seq_inputs], decoder_outputs)

decoder_outputs = decoder([decoder_inputs, encoder_outputs])
model = keras.Model(
    [encoder_inputs, decoder_inputs], decoder_outputs, name="transformer"
)

model.summary()
model.compile(
    optimizer="rmsprop", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)

c:\Users\quang\anaconda3\envs\keras\Lib\site-packages\keras\src\layers\layer.py:982: UserWarning: Layer 'functional_3' (of type Functional) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 64)  │    961,280 │ encoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, None, 64)  │     99,712 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_3        │ (None, None,      │  2,102,488 │ decoder_inputs[0… │
│ (Functional)        │ 15000)            │            │ transformer_enco… │
│                     │                   │            │ not_equal[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,163,480 (12.07 MB)

 Trainable params: 3,163,480 (12.07 MB)

 Non-trainable params: 0 (0.00 B)

---

## Training
The model is trained using the training dataset for 5 epochs.

In [8]:
model.fit(train_ds, epochs=5, validation_data=val_ds)

Epoch 1/5
806/806 ━━━━━━━━━━━━━━━━━━━━ 353s 429ms/step - accuracy: 0.3719 - loss: 5.2144 - val_accuracy: 0.4553 - val_loss: 4.1113
Epoch 2/5
806/806 ━━━━━━━━━━━━━━━━━━━━ 312s 386ms/step - accuracy: 0.4671 - loss: 4.1561 - val_accuracy: 0.5209 - val_loss: 3.5637
Epoch 3/5
806/806 ━━━━━━━━━━━━━━━━━━━━ 298s 370ms/step - accuracy: 0.5112 - loss: 3.7560 - val_accuracy: 0.5585 - val_loss: 3.2677
Epoch 4/5
806/806 ━━━━━━━━━━━━━━━━━━━━ 268s 332ms/step - accuracy: 0.5380 - loss: 3.5237 - val_accuracy: 0.5745 - val_loss: 3.1453
Epoch 5/5
806/806 ━━━━━━━━━━━━━━━━━━━━ 264s 328ms/step - accuracy: 0.5561 - loss: 3.3859 - val_accuracy: 0.5888 - val_loss: 3.0535


---

## Translation Inference
The Finnish vocabulary is converted into a lookup table so token IDs can be turned back into words. A maximum length is set for the generated sentence.

An English sentence is tokenized and passed to the model. The decoder then generates the translation one word at a time, always using the previous output as input for the next step.

Generation starts with a “start” token and stops when an “end” token is produced or the maximum length is reached. After this, the special tokens are removed to form the final sentence.

Finally, a few random test sentences are translated and printed.

In [11]:
fin_vocab = fin_vectorization.get_vocabulary()
fin_index_lookup = dict(zip(range(len(fin_vocab)), fin_vocab))
max_decoded_sentence_length = 20

def decode_sequence(input_sentence):
    tokenized_input_sentence = eng_vectorization([input_sentence])
    
    decoded_sentence = "start" 
    
    for i in range(max_decoded_sentence_length):
        tokenized_target_sentence = fin_vectorization([decoded_sentence])[:, :-1]
        predictions = model([tokenized_input_sentence, tokenized_target_sentence])

        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = fin_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "end":
            break
            
    clean_sentence = decoded_sentence.replace("start ", "").replace(" end", "")
    return clean_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(5):
    input_sentence = random.choice(test_eng_texts)
    translated = decode_sequence(input_sentence)
    print(f"English: {input_sentence}")
    print(f"Translated: {translated}\n")

English: You've probably seen that already.
Translated: olet varma että sinä jo nähnyt sitä

English: I was with Tom.
Translated: olin tomin kanssa

English: I really want to know her name.
Translated: haluan todella kirjan hänen nimensä

English: I can give you the address if you want.
Translated: voin antaa sinulle [UNK] jos haluat

English: It's going to be all right.
Translated: se on menossa koko

